# Quick Start API: One-Call Setup

## Overview

Get from mapping to running instance with a single API call. The `quick_start()` method handles snapshot creation, export, and instance launch automatically.

**What you'll learn:**
- Use `quick_start()` to create resources in one call
- Understand resource hierarchy (mapping → snapshot → instance)
- Verify all resources are properly linked

**Prerequisites:**
- Completed: [06_core_end_to_end_workflows.ipynb](./06_core_end_to_end_workflows.ipynb)
- Understanding of mappings and instances

**Time to complete:** 10 minutes

---

**Test Coverage:**
- quick_start() convenience method
- Automatic snapshot and instance creation
- Resource linking verification

In [ ]:
import os

# Parameters cell - papermill will inject values here
# Note: Uses GRAPH_OLAP_API_URL from environment (set by JupyterHub or local dev)

## Setup

### Test Data Strategy

This test uses quick_start() to create all resources in one call:
1. **Mapping**: Provided as parameter
2. **Snapshot**: Created automatically
3. **Instance**: Created automatically
4. **Cleanup**: Automatic cleanup via cleanup framework

In [ ]:
import sys
import os

print(f"Python version: {sys.version}")
print(f"GRAPH_OLAP_API_URL: {os.environ.get('GRAPH_OLAP_API_URL', 'not set')}")

In [ ]:
from graph_olap import notebook
from graph_olap.exceptions import NotFoundError
from graph_olap.models.mapping import EdgeDefinition, NodeDefinition, PropertyDefinition
from graph_olap.testing import TestPersona
from graph_olap_schemas import WrapperType

print("SDK imports successful")

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

## Connect to SDK

In [ ]:
# Create test context with automatic cleanup
ctx = notebook.test("QuickStartTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

print(f"Connected to {client._config.api_url}")
print(f"Test run ID: {ctx.run_id}")

## Initialize Cleanup Tracking

In [ ]:
# Resources are automatically tracked and cleaned up via ctx
print("Starting Quick Start E2E Test - resources will be cleaned up automatically via atexit")

## 1. Quick Start Tests

### 1.1 Create Mapping for Quick Start

In [ ]:
# Create mapping for quick_start test using ctx.mapping (auto-tracked)
person_node = NodeDefinition(
    label="Person",
    sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
    primary_key={"name": "id", "type": "STRING"},
    properties=[
        PropertyDefinition(name="name", type="STRING"),
        PropertyDefinition(name="age", type="INT64"),
    ]
)

knows_edge = EdgeDefinition(
    type="KNOWS",
    from_node="Person",
    to_node="Person",
    sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
    from_key="from_id",
    to_key="to_id",
    properties=[
        PropertyDefinition(name="since", type="INT64"),
    ]
)

mapping = ctx.mapping(
    description="Mapping for quick_start test",
    node_definitions=[person_node],
    edge_definitions=[knows_edge],
)
MAPPING_ID = mapping.id
MAPPING_NAME = mapping.name

print(f"Created mapping: {MAPPING_NAME} (id={MAPPING_ID})")

### 1.2 Test quick_start() Creates Everything

In [ ]:
# Test: quick_start() creates snapshot and instance
SNAPSHOT_NAME = f"QuickStartTest-Snapshot-{ctx.run_id}"
INSTANCE_NAME = f"QuickStartTest-Instance-{ctx.run_id}"

print("Calling quick_start() - this will create snapshot and instance automatically...")
print(f"  Mapping ID: {MAPPING_ID}")
print(f"  Snapshot name: {SNAPSHOT_NAME}")
print(f"  Instance name: {INSTANCE_NAME}")

instance = client.quick_start(
    mapping_id=MAPPING_ID,
    snapshot_name=SNAPSHOT_NAME,
    instance_name=INSTANCE_NAME,
    wrapper_type=WrapperType.RYUGRAPH,
    wait_timeout=300,  # 5 minutes
)

# Track instance for cleanup via ctx
INSTANCE_ID = instance.id
ctx._track('instance', INSTANCE_ID, INSTANCE_NAME)

print("\nQUICK_START 1.2 PASSED: quick_start() succeeded")
print(f"  Instance: {instance.name} (id={INSTANCE_ID})")
print(f"  Snapshot ID: {instance.snapshot_id}")

### 1.3 Test Instance is Running

In [ ]:
# Test: Instance should be running
assert instance.current_status == "running", f"Expected status 'running', got '{instance.current_status}'"
assert instance.snapshot_id is not None, "Instance should have snapshot_id"
assert instance.instance_url is not None, "Running instance should have URL"

# Track snapshot for cleanup via ctx
SNAPSHOT_ID = instance.snapshot_id
ctx._track('snapshot', SNAPSHOT_ID, SNAPSHOT_NAME)

print("QUICK_START 1.3 PASSED: Instance is running")
print(f"  Snapshot ID: {SNAPSHOT_ID}")
print(f"  Instance URL: {instance.instance_url}")

### 1.4 Test Snapshot Was Created

In [ ]:
# Test: Verify snapshot was created and is ready
snapshot = client.snapshots.get(SNAPSHOT_ID)

assert snapshot is not None, "Snapshot should exist"
assert snapshot.name == SNAPSHOT_NAME, f"Expected snapshot name '{SNAPSHOT_NAME}', got '{snapshot.name}'"
assert snapshot.status == "ready", f"Snapshot should be ready, got '{snapshot.status}'"
assert snapshot.mapping_id == MAPPING_ID, "Snapshot should be linked to mapping"

print("QUICK_START 1.4 PASSED: Snapshot was created")
print(f"  Snapshot: {snapshot.name} (id={SNAPSHOT_ID})")
print(f"  Status: {snapshot.status}")
print(f"  Mapping ID: {snapshot.mapping_id}")

### 1.5 Test Can Query Instance

In [ ]:
# Test: Can connect and query the instance
conn = client.instances.connect(INSTANCE_ID)

# Query node count
node_count = conn.query_scalar("MATCH (n) RETURN count(n)")
assert node_count > 0, "Instance should have nodes"

# Query edge count
edge_count = conn.query_scalar("MATCH ()-[r]->() RETURN count(r)")
assert edge_count > 0, "Instance should have edges"

print("QUICK_START 1.5 PASSED: Can query instance")
print(f"  Nodes: {node_count}")
print(f"  Edges: {edge_count}")

### 1.6 Test All Resources Are Linked

In [ ]:
# Test: Verify resource hierarchy (mapping → snapshot → instance)
assert snapshot.mapping_id == MAPPING_ID, "Snapshot should link to mapping"
assert instance.snapshot_id == SNAPSHOT_ID, "Instance should link to snapshot"

# Verify we can navigate the hierarchy
mapping_from_snapshot = client.mappings.get(snapshot.mapping_id)
assert mapping_from_snapshot.id == MAPPING_ID, "Can navigate snapshot → mapping"

snapshot_from_instance = client.snapshots.get(instance.snapshot_id)
assert snapshot_from_instance.id == SNAPSHOT_ID, "Can navigate instance → snapshot"

print("QUICK_START 1.6 PASSED: All resources are properly linked")
print(f"  Mapping {MAPPING_ID} → Snapshot {SNAPSHOT_ID} → Instance {INSTANCE_ID}")

## Cleanup

In [ ]:
# Cleanup is handled automatically by ctx via atexit
# For interactive use, you can call ctx.cleanup() manually

print("\nCleanup will happen automatically via atexit")

## Summary

In [ ]:
print("\n" + "="*60)
print("QUICK START E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  1. Quick Start Workflow:")
print("    1.1: Create mapping for test")
print("    1.2: quick_start() creates snapshot and instance")
print("    1.3: Instance is running and ready")
print("    1.4: Snapshot was created with correct name")
print("    1.5: Can connect and query instance")
print("    1.6: All resources properly linked (mapping → snapshot → instance)")
print("\nAll resources will be cleaned up automatically via atexit")